# Experiment 2

Compare medium PPO with Frenet and LiDAR across ten paired training roots. Each of the 20 runs receives one million training interactions. The report uses only this matrix; the historical Experiment 2 directory is retained separately.

The protocol is in [EXPERIMENT.md](../docs/EXPERIMENT.md), and the complete interpretation is in [EXPERIMENT_2.md](../docs/EXPERIMENT_2.md). These cells reuse the command-line runner and saved analysis tables.

## Fixed conditions

Training uses stochastic bounded Gaussian policies and randomized starts; evaluation uses `tanh(mean)` from the canonical zero-speed start with frozen normalization. Evaluation interactions do not enter the training budget. Physics, reward, actor initialization, learning rates, critic hidden widths, and normalization remain fixed. All runs use eight CPU environment workers and one Torch intra/inter-op thread. Evaluation occurs every 50k interactions.

Both observations are partial. REINFORCE stops its Monte Carlo return at the 40-second deadline, while A2C/PPO retain a critic bootstrap there. This known mismatch is unchanged in both experiments.

Both actor and critic use `(64,64)` hidden widths. The fixed medium choice comes from the original small/medium/large selection; tiny does not revise it. Roots are `0..9`; validation/test/training-reference contain 16/32/16 circuits. The existing split is reused and is not a new untouched confirmation set.

## Plan the execution

Set `USE_REHEARSAL = True` for the separate reduced-budget rehearsal. The next cell only lists runs. Completed matching results are skipped by execution; incompatible completed results are preserved and refused.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
sys.path.insert(0, str(PROJECT_ROOT / "experiments"))
import experiment_2 as study
from matrix import execute, summarize
from reporting import read_table, show_table, show_figure

USE_REHEARSAL = False
SCALE = study.REHEARSAL if USE_REHEARSAL else study.PROTOCOL
RESULTS_ROOT = SCALE.results_root
ANALYSIS_ROOT = SCALE.analysis_root
SPECIFICATIONS = study.specifications(SCALE)
print(
    f"{len(SPECIFICATIONS)} scheduled runs; budget {SCALE.budget:,}; roots {SCALE.roots}"
)
print(RESULTS_ROOT)
show_table(
    [
        {"run_id": run.run_id, "complete": (run.path / "completion.json").is_file()}
        for run in SPECIFICATIONS
    ]
)


## Execute

This cell trains the pending runs sequentially and can take hours. It never schedules the original Experiment 1 actors when the tiny filter above is retained.

In [ ]:
OUTCOMES = execute(SPECIFICATIONS, contract=study.contract())
FAILURES = summarize(OUTCOMES)
assert FAILURES == 0, "Inspect failed/refused runs before reporting results."


## Regenerate the analysis

Each table and figure below comes from the saved records, with input checksums in `analysis_manifest.json`. Failures remain in the primary results. Lap time is conditional on completion and always includes its denominator.

In [ ]:
MANIFEST = study.analyze(SCALE)
SUMMARIES = read_table(ANALYSIS_ROOT, "run_summaries")
CELLS = read_table(ANALYSIS_ROOT, "cell_summaries")
PAIRED = read_table(ANALYSIS_ROOT, "paired_summaries")
print(f"Analyzed {len(MANIFEST["inputs"])} runs")


### Final performance and reliability

Intervals resample whole training roots. Ten-root intervals use 10,000 seeded resamples (seed 0), with 2.5th and 97.5th percentiles. The 32 test circuits do not multiply the number of independent trained agents.

In [ ]:
CELL_DISPLAY = [
    {
        "condition": "/".join(
            str(row[key])
            for key in ("algorithm", "actor_name", "observation_type")
            if key in row
        ),
        "roots": row["root_count"],
        "laps": f"{row['completed_lap_count']}/{row['completed_lap_denominator']}",
        "mean_return": row["final_mean_return"]["mean"],
        "return_SD": row["final_mean_return"]["sample_standard_deviation"],
        "return_95_low": row["final_mean_return"]["confidence_interval_low"],
        "return_95_high": row["final_mean_return"]["confidence_interval_high"],
        "mean_progress": row["final_mean_progress"]["mean"],
        "mean_lap_time": (
            None
            if row["completed_lap_time"] is None
            else row["completed_lap_time"]["mean"]
        ),
        "threshold_roots": row["converged_root_count"],
    }
    for row in CELLS
]
show_table(CELL_DISPLAY)
show_figure(ANALYSIS_ROOT, "task_outcomes")


### Learning curves and common budgets

Thin curves show individual roots and thick curves show root means in the comparison panels. Endpoint similarity does not imply similar learning curves. Compare widths within an algorithm and algorithms within a width. The common-budget table contains only the declared 250k, 500k, 750k, 1M and (where available) 2M boundaries.

In [ ]:
show_figure(ANALYSIS_ROOT, "learning_curves")
show_table(
    read_table(ANALYSIS_ROOT, "common_budget_outcomes"),
    columns=[
        "algorithm",
        "actor_name",
        "observation_type",
        "root_identity",
        "training_interactions",
        "mean_return",
        "completion_rate",
    ],
    limit=40,
)


### Threshold cost and late stability

Attainment is the first of three consecutive qualifying evaluations. Confirmation costs another 100k interactions at the reported cadence. Experiment 1 requires a completed lap within 34 seconds. Experiment 2 requires 12/16 validation completions and median progress at least 0.95; the median condition is redundant once 12 circuits complete. Attainment does not guarantee lasting convergence. Censored roots keep missing attainment times; capped values are not observed convergence. Training duration counts collection plus optimization, not full-budget end-to-end runtime. Late stability uses `0.8 * budget < interactions <= budget` and supplements the unchanged final result.

In [ ]:
show_table(
    SUMMARIES,
    columns=[
        "algorithm",
        "actor_name",
        "observation_type",
        "root_identity",
        "converged",
        "censored",
        "convergence_interactions",
        "confirmation_interactions",
        "episodes_to_convergence",
        "episodes_to_confirmation",
        "convergence_duration",
        "confirmation_duration",
        "late_completion_rate",
        "late_mean_return",
    ],
)
show_figure(ANALYSIS_ROOT, "convergence_resources")


### Paired contrasts

Positive differences favour the first named condition. Shared root identities define the pairing, but do not guarantee cancellation of random variation. Intervals including zero do not establish equivalence; multiple comparisons are exploratory.

In [ ]:
show_table(
    [
        {
            "algorithm": row["algorithm"],
            "actor": row["actor_name"],
            "contrast": row["contrast"],
            "roots": row["root_count"],
            "return_difference": row["final_mean_return"]["mean"],
            "return_95_low": row["final_mean_return"]["confidence_interval_low"],
            "return_95_high": row["final_mean_return"]["confidence_interval_high"],
            "completion_difference": row["final_completion_rate"]["mean"],
            "completion_95_low": row["final_completion_rate"][
                "confidence_interval_low"
            ],
            "completion_95_high": row["final_completion_rate"][
                "confidence_interval_high"
            ],
        }
        for row in PAIRED
    ]
)


### Learned controls

Control summaries include all roots and failures. Steps have equal weight within a circuit, circuits equal weight within a root, and roots equal weight in comparisons. The table includes observed coverage. Braking means requested throttle below zero; near-zero throttle means absolute value at most 0.05. Steering sign reversals ignore requested magnitudes at most 0.05; saturation means magnitude at least 0.9. Requested steering is different from the actual wheel angle and grip-limited turning.

Traces illustrate fixed root 0 (test circuit 0 in Experiment 2). Their distance is pre-action, aligned with speed and controls. Grey shading marks observed geometric corners; it does not fill missing coverage after failure. Straight curvature has its own group at tolerance `1e-8`, and curved groups use unique positive circuit-geometry quartile boundaries. No control trace establishes a causal mechanism by itself.

In [ ]:
show_table(
    read_table(ANALYSIS_ROOT, "root_controls"),
    columns=[
        "algorithm",
        "actor_name",
        "observation_type",
        "root_identity",
        "circuit_count",
        "completed_circuit_count",
        "coverage",
        "mean_speed",
        "braking_fraction",
        "mean_throttle",
        "throttle_standard_deviation",
        "mean_steering_change",
        "steering_reversals_per_time",
    ],
)
show_figure(ANALYSIS_ROOT, "control_summaries")
show_figure(ANALYSIS_ROOT, "control_traces")
show_figure(ANALYSIS_ROOT, "curvature_controls")


### Generalization and circuit variation

Compare each root across the training-reference, validation and test splits. The test result describes this frozen generator family. A small gap does not establish absence of a generalization gap. Per-circuit differences are descriptive, not independent training replicates. Different episode lengths can produce unequal total training-circuit exposure.

In [ ]:
show_table(read_table(ANALYSIS_ROOT, "final_split_summaries"))
show_table(read_table(ANALYSIS_ROOT, "generalization_gaps"))
show_figure(ANALYSIS_ROOT, "circuit_geometry")


### Optimization and resources

Low critic explained variance does not show that value accuracy is irrelevant. Pre-squash Gaussian scale is not the variation of bounded actions, and reaching its bound does not establish the cause of a plateau. Process-lifetime peak memory excludes workers and may carry across runs; it is not an isolated per-run memory measurement.

In [ ]:
show_figure(ANALYSIS_ROOT, "optimization_diagnostics")
show_table(
    SUMMARIES,
    columns=[
        "algorithm",
        "actor_name",
        "observation_type",
        "root_identity",
        "collection_duration",
        "optimization_duration",
        "evaluation_duration",
        "training_duration",
        "end_to_end_duration",
        "collection_throughput",
    ],
)
